In the previous notebook 1modeling_exerpiments.ipynb, we found that the Prophet model performed well in terms of forecasting rat sightings by day citywide. In this notebook, we will do some more feature engineering and hyperparameter tuning to obtain a better optimal model.

Because we wish this to be reusable, we will write things for Prophet and NeuralProphet separately. 

# Import Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging


Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


# Prophet

## Import the data

In [2]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2025-02-28.
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to STATEN ISLAND

rs = rs[rs['borough']=='STATEN ISLAND']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


## Prepare Prophet

In [3]:
date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [4]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [5]:
rs_saved = rs.copy()
df = rs.copy()

## Grid Search for Hyperparameter Tuning for Prophet

In [6]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

In [8]:
# This code block is a grid search for hyperparameters.
# To tune for hyperparameters, add more possible parameters to the dictionary below and add more values to it.
# So far, the I've been able to get is {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5}

init_days = f'{len(rs)-(14*14)} days'
cv_period = '14 days'
forecast_horizon = '14 days'

param_grid = {  
    'changepoint_prior_scale': [0.5, 0.1, 0.001, ],
    'seasonality_prior_scale': [1, 2, 5, 8],
}

# Generate all combinations of parameters
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
rmses = []  # Store the RMSEs for each params here
performance = []

# Use cross validation to evaluate all parameters
for params in all_params:
    params['holidays'] = holidays
    m = Prophet(**params).fit(df)  # Fit model with given params
    df_cv = cross_validation(m, initial = init_days, period=cv_period, horizon = forecast_horizon)
    df_p = performance_metrics(df_cv, rolling_window=14)
    performance.append(df_p)
    rmses.append(df_p['rmse'].values[0])

# Find the best parameters
tuning_results = pd.DataFrame(all_params)
tuning_results['rmse'] = rmses

best_params = all_params[np.argmin(rmses)]

15:49:39 - cmdstanpy - INFO - Chain [1] start processing
15:49:39 - cmdstanpy - INFO - Chain [1] done processing


  0%|          | 0/49 [00:00<?, ?it/s]

15:49:40 - cmdstanpy - INFO - Chain [1] start processing
15:49:40 - cmdstanpy - INFO - Chain [1] done processing
15:49:40 - cmdstanpy - INFO - Chain [1] start processing
15:49:40 - cmdstanpy - INFO - Chain [1] done processing
15:49:41 - cmdstanpy - INFO - Chain [1] start processing
15:49:41 - cmdstanpy - INFO - Chain [1] done processing
15:49:41 - cmdstanpy - INFO - Chain [1] start processing
15:49:41 - cmdstanpy - INFO - Chain [1] done processing
15:49:41 - cmdstanpy - INFO - Chain [1] start processing
15:49:42 - cmdstanpy - INFO - Chain [1] done processing
15:49:42 - cmdstanpy - INFO - Chain [1] start processing
15:49:42 - cmdstanpy - INFO - Chain [1] done processing
15:49:42 - cmdstanpy - INFO - Chain [1] start processing
15:49:42 - cmdstanpy - INFO - Chain [1] done processing
15:49:43 - cmdstanpy - INFO - Chain [1] start processing
15:49:43 - cmdstanpy - INFO - Chain [1] done processing
15:49:43 - cmdstanpy - INFO - Chain [1] start processing
15:49:43 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/49 [00:00<?, ?it/s]

15:50:00 - cmdstanpy - INFO - Chain [1] start processing
15:50:00 - cmdstanpy - INFO - Chain [1] done processing
15:50:01 - cmdstanpy - INFO - Chain [1] start processing
15:50:01 - cmdstanpy - INFO - Chain [1] done processing
15:50:01 - cmdstanpy - INFO - Chain [1] start processing
15:50:01 - cmdstanpy - INFO - Chain [1] done processing
15:50:01 - cmdstanpy - INFO - Chain [1] start processing
15:50:01 - cmdstanpy - INFO - Chain [1] done processing
15:50:02 - cmdstanpy - INFO - Chain [1] start processing
15:50:02 - cmdstanpy - INFO - Chain [1] done processing
15:50:02 - cmdstanpy - INFO - Chain [1] start processing
15:50:02 - cmdstanpy - INFO - Chain [1] done processing
15:50:03 - cmdstanpy - INFO - Chain [1] start processing
15:50:03 - cmdstanpy - INFO - Chain [1] done processing
15:50:03 - cmdstanpy - INFO - Chain [1] start processing
15:50:03 - cmdstanpy - INFO - Chain [1] done processing
15:50:03 - cmdstanpy - INFO - Chain [1] start processing
15:50:04 - cmdstanpy - INFO - Chain [1]

...

15:52:33 - cmdstanpy - INFO - Chain [1] start processing
15:52:33 - cmdstanpy - INFO - Chain [1] done processing
15:52:33 - cmdstanpy - INFO - Chain [1] start processing
15:52:33 - cmdstanpy - INFO - Chain [1] done processing
15:52:33 - cmdstanpy - INFO - Chain [1] start processing
15:52:33 - cmdstanpy - INFO - Chain [1] done processing
15:52:34 - cmdstanpy - INFO - Chain [1] start processing
15:52:34 - cmdstanpy - INFO - Chain [1] done processing
15:52:34 - cmdstanpy - INFO - Chain [1] start processing
15:52:34 - cmdstanpy - INFO - Chain [1] done processing
15:52:34 - cmdstanpy - INFO - Chain [1] start processing
15:52:34 - cmdstanpy - INFO - Chain [1] done processing
15:52:35 - cmdstanpy - INFO - Chain [1] start processing
15:52:35 - cmdstanpy - INFO - Chain [1] done processing
15:52:35 - cmdstanpy - INFO - Chain [1] start processing
15:52:35 - cmdstanpy - INFO - Chain [1] done processing
15:52:35 - cmdstanpy - INFO - Chain [1] start processing
15:52:35 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/49 [00:00<?, ?it/s]

15:52:51 - cmdstanpy - INFO - Chain [1] start processing
15:52:51 - cmdstanpy - INFO - Chain [1] done processing
15:52:52 - cmdstanpy - INFO - Chain [1] start processing
15:52:52 - cmdstanpy - INFO - Chain [1] done processing
15:52:52 - cmdstanpy - INFO - Chain [1] start processing
15:52:52 - cmdstanpy - INFO - Chain [1] done processing
15:52:52 - cmdstanpy - INFO - Chain [1] start processing
15:52:52 - cmdstanpy - INFO - Chain [1] done processing
15:52:53 - cmdstanpy - INFO - Chain [1] start processing
15:52:53 - cmdstanpy - INFO - Chain [1] done processing
15:52:53 - cmdstanpy - INFO - Chain [1] start processing
15:52:53 - cmdstanpy - INFO - Chain [1] done processing
15:52:53 - cmdstanpy - INFO - Chain [1] start processing
15:52:53 - cmdstanpy - INFO - Chain [1] done processing
15:52:54 - cmdstanpy - INFO - Chain [1] start processing
15:52:54 - cmdstanpy - INFO - Chain [1] done processing
15:52:54 - cmdstanpy - INFO - Chain [1] start processing
15:52:54 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/49 [00:00<?, ?it/s]

15:53:10 - cmdstanpy - INFO - Chain [1] start processing
15:53:10 - cmdstanpy - INFO - Chain [1] done processing
15:53:10 - cmdstanpy - INFO - Chain [1] start processing
15:53:10 - cmdstanpy - INFO - Chain [1] done processing
15:53:10 - cmdstanpy - INFO - Chain [1] start processing
15:53:10 - cmdstanpy - INFO - Chain [1] done processing
15:53:11 - cmdstanpy - INFO - Chain [1] start processing
15:53:11 - cmdstanpy - INFO - Chain [1] done processing
15:53:11 - cmdstanpy - INFO - Chain [1] start processing
15:53:11 - cmdstanpy - INFO - Chain [1] done processing
15:53:11 - cmdstanpy - INFO - Chain [1] start processing
15:53:11 - cmdstanpy - INFO - Chain [1] done processing
15:53:12 - cmdstanpy - INFO - Chain [1] start processing
15:53:12 - cmdstanpy - INFO - Chain [1] done processing
15:53:12 - cmdstanpy - INFO - Chain [1] start processing
15:53:12 - cmdstanpy - INFO - Chain [1] done processing
15:53:12 - cmdstanpy - INFO - Chain [1] start processing
15:53:12 - cmdstanpy - INFO - Chain [1]

In [9]:
best_params

{'changepoint_prior_scale': 0.1,
 'seasonality_prior_scale': 5,
 'holidays':        holiday         ds  lower_window  upper_window
 0   federal_us 2020-01-01             0             1
 1   federal_us 2020-01-20             0             1
 2   federal_us 2020-02-17             0             1
 3   federal_us 2020-05-25             0             1
 4   federal_us 2020-07-03             0             1
 5   federal_us 2020-09-07             0             1
 6   federal_us 2020-10-12             0             1
 7   federal_us 2020-11-11             0             1
 8   federal_us 2020-11-26             0             1
 9   federal_us 2020-12-25             0             1
 10  federal_us 2021-01-01             0             1
 11  federal_us 2021-01-18             0             1
 12  federal_us 2021-02-15             0             1
 13  federal_us 2021-05-31             0             1
 14  federal_us 2021-06-18             0             1
 15  federal_us 2021-07-05             0    

In [10]:
new_performance = pd.concat(performance, ignore_index=True)

# Round numeric columns for readability
numeric_cols = ["mse", "rmse", "mae", "mape", "mdape", "smape", "coverage"]
new_performance[numeric_cols] = new_performance[numeric_cols].round(4)


new_performance

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,14 days,3.1265,1.7682,1.2265,0.6454,0.4568,0.4907,0.8897
1,14 days,3.1287,1.7688,1.2262,0.6446,0.4541,0.4907,0.8878
2,14 days,3.1238,1.7674,1.2264,0.6454,0.4564,0.4907,0.8878
3,14 days,3.1199,1.7663,1.2257,0.6451,0.4569,0.4904,0.8839
4,14 days,3.0493,1.7462,1.2193,0.6493,0.4539,0.4880,0.8917
5,14 days,3.0516,1.7469,1.2194,0.6490,0.4535,0.4880,0.8859
6,14 days,3.0482,1.7459,1.2194,0.6493,0.4536,0.4880,0.8859
7,14 days,3.0496,1.7463,1.2195,0.6493,0.4530,0.4880,0.8878
8,14 days,3.1928,1.7868,1.3256,0.7959,0.4911,0.5170,0.8646
9,14 days,3.1978,1.7882,1.3300,0.8019,0.4889,0.5172,0.8607


## Train the Model

In [11]:
m = Prophet(**best_params)
m.add_country_holidays(country_name='US')
m.fit(df)
future = m.make_future_dataframe(periods=14)
forecast = m.predict(future)

15:53:28 - cmdstanpy - INFO - Chain [1] start processing
15:53:28 - cmdstanpy - INFO - Chain [1] done processing


## Plots and Evaluation of the Model

In [12]:
fig1 = plot_plotly(m, forecast)
fig1.show()

fig2 = plot_components_plotly(m, forecast)
fig2.show()

WARNING - (py.warnings._showwarnmsg) - c:\Users\daoke\anaconda3\Lib\site-packages\_plotly_utils\basevalidators.py:105: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  v = v.dt.to_pydatetime()



WARNING - (py.warnings._showwarnmsg) - c:\Users\daoke\anaconda3\Lib\site-packages\_plotly_utils\basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result


WARNING - (py.warnings._showwarnmsg) - c:\Users\daoke\anaconda3\Lib\site-packages\_plotly_utils\basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result


WARNING - (py.warnings._showwarnmsg) - c:\Users\daoke\anaconda3\Lib\site-packages\_plotly_utils\basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime ob

# Neural Prophet

## Load and Prepare Data

In [13]:
np.NaN = np.nan


# the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# the errors that do show up are not all that important and a lot is due to outdated packages.
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("neuralprophet").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("NP").setLevel(logging.ERROR)

In [14]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data 
rs = rs[rs['created_date']<'2025-02-28']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to STATEN ISLAND

rs = rs[rs['borough']=='STATEN ISLAND']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

rs

,ds,y
4,2020-01-01,1
9,2020-01-02,1
14,2020-01-03,1
27,2020-01-06,4
32,2020-01-07,2
...,...,...
8893,2025-02-21,1
8902,2025-02-23,5
8907,2025-02-24,3
8912,2025-02-25,1


In [15]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28" 

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [16]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)


regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

## Optuna Hyperparameter Tuning

In [17]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


def objective(trial):
    regressor_lags = {
        'apparent_temperature_max': trial.suggest_int('lag_temp_max', 1, 60),
        'apparent_temperature_min': trial.suggest_int('lag_temp_min', 1, 60),
        'snowfall_sum': trial.suggest_int('lag_snowfall', 1, 7),
    }
    n_lags = trial.suggest_int('n_lags', 1, 60)
    epochs = trial.suggest_int('epochs', 10, 250)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 1, log=True)
    batch_size = trial.suggest_int('batch_size', 12, 248)
    ar_reg = trial.suggest_float('ar_reg', 0.5, 3)
    fold_rmses = []
    for i, (train_idx, test_idx) in enumerate(tscv.split(rs)):

        train = rs.iloc[train_idx].copy()
        test = rs.iloc[test_idx].copy()
        
        existing_regressors = [col for col in regressed_features if col in train.columns]
        train = train.dropna(subset=["y"] + existing_regressors)
        test = test.dropna(subset=existing_regressors)
        
        # Skip fold if too few rows
        if len(train) < 20 or len(test) < 1:
            continue
        
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            n_lags=n_lags,
            epochs=epochs,
            ar_reg = ar_reg,
            accelerator="auto",   # uses GPU if available
            learning_rate=learning_rate,
            batch_size=batch_size
        )
        model.add_country_holidays(country_name="US")
        for col in existing_regressors:
            model.add_lagged_regressor(col, n_lags=regressor_lags[col])
        
        model.fit(train, freq="D", progress="off")
        future = pd.concat([
            train[['ds','y'] + existing_regressors],
            test[['ds','y']].merge(wd[['ds'] + existing_regressors], on="ds", how="left")
        ])
        future = future.dropna(subset=existing_regressors)
        forecast = model.predict(future)
        
        y_pred = forecast["yhat1"].iloc[-len(test):].values
        y_true = test["y"].values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(rmse)
        
    return np.mean(fold_rmses)

study = optuna.create_study(
    direction="minimize",
    study_name="model_neural",
    storage="sqlite:///model_neural.db",
    load_if_exists=True
)
study.optimize(objective, n_trials=1)  # adjust n_trials as needed



best_params = study.best_params

print("Best Parameters", best_params)
print("Best RMSE:", study.best_value)

[I 2026-03-14 15:53:33,438] A new study created in RDB with name: model_neural


Training: 0it [00:00, ?it/s]

Predicting: 18it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 18it [00:00, ?it/s]

...

Predicting: 24it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 24it [00:00, ?it/s]

[I 2026-03-14 16:02:06,991] Trial 0 finished with value: 1.3496383985765343 and parameters: {'lag_temp_max': 56, 'lag_temp_min': 36, 'lag_snowfall': 2, 'n_lags': 18, 'epochs': 108, 'learning_rate': 0.5614094098581325, 'batch_size': 77, 'ar_reg': 2.2564227941767063}. Best is trial 0 with value: 1.3496383985765343.


Best Parameters {'lag_temp_max': 56, 'lag_temp_min': 36, 'lag_snowfall': 2, 'n_lags': 18, 'epochs': 108, 'learning_rate': 0.5614094098581325, 'batch_size': 77, 'ar_reg': 2.2564227941767063}
Best RMSE: 1.3496383985765343


In [18]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to STATEN ISLAND

rs = rs[rs['borough']=='STATEN ISLAND']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

wd = pd.DataFrame(data["daily"])
wd["date"] = pd.to_datetime(wd["time"])
wd = wd.set_index("date")

KeyError: 'daily'

## Evaluate the Model

In [ ]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min','snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']


results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    train = train.dropna(subset=["y"])

    test = rs.iloc[test_index].copy()


    model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          learning_rate = best_params['learning_rate'],
                          epochs = best_params['epochs'],
                          n_lags= best_params['n_lags'],
                          ar_reg=best_params['ar_reg'],
                          accelerator="auto",   # uses GPU if available
                          batch_size= best_params['batch_size']
                          )
    model = model.add_country_holidays(country_name="US")
    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])
        
    model.fit(train, freq="D", progress="off")

    # build dataframe containing future regressors
    future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_true = test["y"].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse, "mape": mape})

neural_prophet_results_df = pd.DataFrame(results)
neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]

Training: 0it [00:00, ?it/s]

Predicting: 10it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 10it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

...

Predicting: 13it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 13it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 13it [00:00, ?it/s]

## Neural Prophet's Results

In [ ]:
neural_prophet_results_df

,fold,rmse,mape
0,0,2.837809,0.189877
1,1,4.389223,0.252642
2,2,4.222367,0.357147
3,3,5.742150,0.314968
4,4,5.111527,0.386820
5,5,5.696702,0.291639
6,6,6.017226,0.256127
7,7,4.211135,0.222551
8,8,3.919665,0.270639
9,9,4.670872,0.183099
